In [1]:
import os
import sys
import tomllib
from datetime import date
from pathlib import Path

os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

# Load local path configuration.
# Copy config/config.example.toml → config/config.toml and adjust paths if needed.
_cfg_file = Path("config/config.toml")
_project_root = _cfg_file.parent.parent

with _cfg_file.open("rb") as _f:
    _cfg = tomllib.load(_f)

_src_path = (_project_root / _cfg["paths"]["src_path"]).resolve()
_data_path = (_project_root / _cfg["paths"]["data_path"]).resolve()

sys.path.insert(0, str(_src_path))

from config.settings import ETLConfig
from utils.spark import get_spark

spark = get_spark()
spark.sparkContext.setLogLevel("WARN")
config = ETLConfig(data_path=_data_path)

In [2]:
from jobs.daily_load import run_daily_load
from jobs.initial_load import run_initial_load

initial = run_initial_load(spark, config)
print("Initial load OK")

DATE_DEBUT = date(2026, 4, 29)
DATE_FIN = date(2026, 5, 7)
daily = run_daily_load(spark, config, DATE_DEBUT, DATE_FIN, initial)
print("Daily load OK")

Initial load OK
Daily load OK


In [18]:
print(f"DIM_DATE          : {initial.dim_date.count()} lignes")
print(f"  {initial.dim_date.dtypes}")
print(f"DIM_TRANSPORT_TYPE: {initial.dim_transport_type.count()} lignes")
print(f"  {initial.dim_transport_type.dtypes}")
print(f"DIM_EQUIPMENT     : {initial.dim_equipment.count()} lignes")
print(f"  {initial.dim_equipment.dtypes}")
print(f"DIM_CITY (initial): {initial.dim_city.count()} villes")
print(f"  {initial.dim_city.dtypes}")
print(f"DIM_STAFF         : {initial.dim_staff.count()} employés")
print(f"  {initial.dim_staff.dtypes}")

DIM_DATE          : 730 lignes
  [('SK_DATE', 'bigint'), ('DATE_ISO', 'date'), ('YEAR', 'int'), ('MONTH', 'int'), ('DAY', 'int')]
DIM_TRANSPORT_TYPE: 5 lignes
  [('SK_TRANSPORT_TYPE', 'bigint'), ('TRANSPORT_NAME', 'string'), ('CO2_FACTOR_KG_PER_KM', 'double')]
DIM_EQUIPMENT     : 80 lignes
  [('SK_EQUIPMENT', 'bigint'), ('TYPE', 'string'), ('MODEL', 'string'), ('CO2_IMPACT_KG_REF', 'double')]
DIM_CITY (initial): 6 villes
  [('SK_CITY', 'bigint'), ('CITY_NAME', 'string'), ('COUNTRY_ISO2', 'string'), ('IS_ORG_SITE', 'boolean'), ('TIMEZONE_IANA', 'string')]
DIM_STAFF         : 20572 employés
  [('SK_STAFF', 'bigint'), ('NK_STAFF', 'string'), ('LAST_NAME', 'string'), ('FIRST_NAME', 'string'), ('JOB_TITLE', 'string'), ('BIRTH_DATE', 'date'), ('SK_SITE', 'bigint')]
